# Bivariate and multivariate analysis

Run each cell in order with **Shift + Enter**. Each step matches a section of the Note.
Every chart is drawn with Plotly: hover over a point, bar or cell to read its exact values; drag to zoom; double-click to reset.

Data (all in `data/`): `tips.csv` (restaurant bills), `titanic_train.csv` (passengers), `flights.csv` (monthly air passengers), `iris.csv` (flowers).

## 1. Load the four datasets

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
from scipy.stats import gaussian_kde
from scipy.cluster.hierarchy import linkage, leaves_list

# the same colours as the Note's figures
BLUE, ORANGE, GREEN, RED = "#4C78A8", "#F58518", "#54A24B", "#E45756"
pd.__version__

In [ ]:
# tips, flights and iris ship with seaborn (sns.load_dataset); we keep local copies
tips = pd.read_csv("data/tips.csv")
titanic = pd.read_csv("data/titanic_train.csv")
flights = pd.read_csv("data/flights.csv")
iris = pd.read_csv("data/iris.csv")
tips.head()

In [ ]:
flights.head()

In [ ]:
iris.head()

## 2. Scatter plot (numerical + numerical)

In [ ]:
# one dot per bill: total bill on x, tip on y
px.scatter(tips, x="total_bill", y="tip")

In [ ]:
# multivariate: colour = sex, marker = smoker, dot size = party size (five columns in one plot)
px.scatter(tips, x="total_bill", y="tip", color="sex", symbol="smoker", size="size",
           color_discrete_map={"Male": BLUE, "Female": ORANGE})

## 3. Bar plot (numerical + categorical)

In [ ]:
# the bar plot shows the mean age of each class; this is the number behind each bar
titanic.groupby("Pclass")["Age"].mean().round(1)

In [ ]:
# mean age per class and sex (hue = sex), as a table and then as bars
mean_age = titanic.groupby(["Pclass", "Sex"], as_index=False)["Age"].mean()
fig = px.bar(mean_age, x="Pclass", y="Age", color="Sex", barmode="group",
             color_discrete_map={"male": BLUE, "female": ORANGE})
fig.update_xaxes(type="category")
fig

## 4. Box plot (numerical + categorical)

In [ ]:
# age by sex, one box per outcome (hue = Survived)
titanic["Outcome"] = titanic["Survived"].map({0: "died", 1: "survived"})
px.box(titanic, x="Sex", y="Age", color="Outcome",
       color_discrete_map={"died": RED, "survived": GREEN})

## 5. KDE plot (numerical + categorical)

In [ ]:
# split the ages by outcome, dropping missing ages
died = titanic.loc[titanic["Survived"] == 0, "Age"].dropna()
survived = titanic.loc[titanic["Survived"] == 1, "Age"].dropna()

# one smooth density curve per group (the old sns.distplot(..., hist=False))
# gaussian_kde builds the curve; we read it at 200 ages from 0 to 80
ages = np.linspace(0, 80, 200)
curves = pd.DataFrame({"Age": ages,
                       "died": gaussian_kde(died)(ages),
                       "survived": gaussian_kde(survived)(ages)})
px.line(curves, x="Age", y=["died", "survived"], labels={"value": "Density", "variable": ""},
        color_discrete_map={"died": RED, "survived": GREEN})

## 6. Heatmap of a crosstab (categorical + categorical)

In [ ]:
# count passengers for every (class, outcome) pair
ct = pd.crosstab(titanic["Pclass"], titanic["Survived"])
ct

In [ ]:
# the same table, coloured by size
px.imshow(ct, text_auto=True, color_continuous_scale="Blues", aspect="auto")

In [ ]:
# survival rate (%) per class: the mean of a 0/1 column is the share of 1s
(titanic.groupby("Pclass")["Survived"].mean() * 100).round(1)

In [ ]:
# the same for sex and port of boarding
for col in ["Sex", "Embarked"]:
    print((titanic.groupby(col)["Survived"].mean() * 100).round(1), "\n")

In [ ]:
# a series plots straight to a bar chart
rate = titanic.groupby("Embarked")["Survived"].mean() * 100
px.bar(rate, labels={"value": "Survived (%)"})

## 7. Clustermap (categorical + categorical)

In [ ]:
def clustered(table):
    """Reorder rows and columns so similar ones sit together (what a clustermap does)."""
    rows = leaves_list(linkage(table.values, method="average"))
    cols = leaves_list(linkage(table.values.T, method="average"))
    return table.iloc[rows, cols]

# crosstab of parents/children aboard vs survival, rows grouped by similarity
parch = pd.crosstab(titanic["Parch"], titanic["Survived"])
px.imshow(clustered(parch), text_auto=True, color_continuous_scale="Blues", aspect="auto")

In [ ]:
# the same idea for siblings/spouses aboard
sibsp = pd.crosstab(titanic["SibSp"], titanic["Survived"])
px.imshow(clustered(sibsp), text_auto=True, color_continuous_scale="Blues", aspect="auto")

## 8. Pair plot (many numerical columns)

In [ ]:
# every pair of the four measurements, coloured by species
cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
fig = px.scatter_matrix(iris, dimensions=cols, color="species",
                        color_discrete_sequence=[BLUE, ORANGE, GREEN], height=750)
fig.update_traces(diagonal_visible=False, marker_size=4)
fig

## 9. Line plot (numerical over time)

In [ ]:
# total passengers per year: group by year, add up the 12 months
yearly = flights.groupby("year", as_index=False)["passengers"].sum()
yearly

In [ ]:
px.line(yearly, x="year", y="passengers", markers=True)

## 10. Pivot table, heatmap and clustermap of the flights

In [ ]:
# month order is lost in a CSV (it sorts alphabetically), so we set it back
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# one row per month, one column per year, passengers in the cells
table = flights.pivot_table(values="passengers", index="month", columns="year")
table = table.reindex(months)
table

In [ ]:
px.imshow(table, color_continuous_scale="Blues", aspect="auto")

In [ ]:
# clustermap: similar months and similar years moved next to each other
px.imshow(clustered(table), text_auto=True, color_continuous_scale="Blues", aspect="auto", height=550)